# 📝 과제 02: 택배 배송 지연 대응 팀의 실패 처리

## 만들 결과

가상 택배사 '바로택배'(수업용 가상 회사)의 **배송 지연 보상 팀**이 외부 API 실패에도 멈추지 않고 끝까지 처리하게 만듭니다. 조회 워커 셋이 배송 추적·날씨·보상 규정 API를 동시에 조회하고, 보상액을 계산해 안내문을 쓰고 검수한 뒤 쿠폰을 발급합니다.

- 일시 오류는 다시 시도하고, 응답 없는 호출은 타임아웃으로 끊고, 영구 오류는 기록만 남기고 나머지 결과로 이어 갑니다.
- 조회가 실패한 뒤 이 소포를 계속 처리할지(완료·축소 진행), 담당자에게 넘길지(보류), 멈출지(중단)를 코드로 판단합니다.
- 보상액 계산이 실패하면 기본 보상액으로 돌아가는 폴백 경로를 만들고, 안내문 검수의 같은 반려 반복은 사람에게 넘깁니다.
- 쿠폰 발급 응답이 끊기면 **이어 하기**(오류로 멈춘 실행을 같은 `thread_id`로 멈춘 단계부터 다시 시작하는 것)로 발급 단계만 다시 실행하고, 쿠폰이 두 장 나가지 않게 멱등 키를 씁니다.
- 소포 3건을 실제로 실행해 `assignment02_runs.json`에 저장하고 Langfuse에서 실패 흔적을 읽습니다.

교안 02(핑퐁·재시도·타임아웃)와 교안 03(부분 실패·폴백·이어 하기)의 장치를 적용합니다.

<img src="images/assignment02_overview.png" width="1050" alt="조회 워커 셋이 Send로 동시에 실행되고, 진행 판단이 보상액 계산이나 사람 확인 요청, END로 보내며, 보상액 계산 뒤 안내문 작성과 검수가 반려되면 다시 쓰는 사이클을 이루며, 통과하면 쿠폰 발급, 같은 사유 반려가 반복되거나 반려 수 상한에 닿으면 사람 확인 요청으로 끝납니다. 보상액 계산이 실패하면 error_handler 폴백으로 안내문 작성에 갑니다.">

## 준비: 데이터와 제공 코드

- **자료:** `data/assignment02_*.json` 4개. 지연 소포 3건(`P-7001`~`P-7003`), 배송 추적·날씨·보상 규정 API가 돌려줄 값입니다(수업용 가상 자료).
- **순서:** 재시도 정책 -> 조회 워커 부분 실패 격리 -> 진행 판단 -> 폴백 핸들러 -> 핑퐁 가드 -> 멱등 발급 -> 팀 그래프 조립 -> 이어 하기 -> 실제 실행 -> 서술
- **제공:** 아래 준비 셀 두 개(환경 설정, 자료와 외부 API 흉내 `track_parcel`·`fetch_weather`·`fetch_policy`·`issue_coupon`, 호출 횟수 `CALLS`, `reset_calls`)와 문항마다 쓰는 팀 State, `REQUIRED_ITEMS`, `quote`, `writer`·`reviewer`, `escalate`, `dispatch_checks`, `make_input`, `make_config`, `count_calls`
- **작성:** `[작성]` 셀의 `...` 자리에 번호 주석대로 코드를 넣고, 바로 아래 자가채점 셀을 실행하세요. 자가채점은 모델과 Langfuse 서버를 호출하지 않습니다. 9번만 모델을 호출하고 Langfuse에 trace를 보냅니다(Langfuse 키 필요).

| 외부 API 흉내 | 일부러 넣은 실패 |
|---|---|
| `track_parcel(parcel_id)` | 세 번 중 앞의 두 번은 `ConnectionError`(일시 오류) |
| `fetch_weather(region)` | async 함수. 홀수 번째 호출은 20초 동안 응답 없음 |
| `fetch_policy(kind)` | 냉장 규정은 응답 형식이 바뀌어 늘 `KeyError`(영구 오류) |
| `issue_coupon(parcel_id, amount, key)` | `FAULTS["coupon_response_lost"]`가 `True`이면 쿠폰을 만든 뒤 응답이 끊김(`ConnectionError`) |

In [ ]:
# 환경 설정: 라이브러리, 경로, API 키, 답변 모델
import os
import json
import time
import asyncio
from datetime import datetime
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from IPython.display import Image, Markdown
# 그래프 정의, 실패 처리, 관측, 자가채점에 필요한 클래스와 함수를 불러옵니다.
import openai
from uuid import uuid4
import re
from unittest.mock import Mock, patch
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, Send, RetryPolicy
from langgraph.errors import NodeError, NodeTimeoutError
from langgraph.checkpoint.memory import InMemorySaver
from langfuse import get_client
from langfuse.langchain import CallbackHandler

# material_dir는 data·output이 있는 단원 폴더입니다. 정답에서는 ".."을 사용합니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 기존 값은 유지합니다.
load_dotenv(material_dir.resolve().parent / ".env")
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)
print("답변 모델:", model_name)

In [ ]:
# JSON 읽기·저장 함수와 과제 자료입니다.
def read_json(name):
    """data 폴더의 JSON 파일을 읽어 파이썬 객체로 반환합니다."""
    return json.loads((data_dir / name).read_text(encoding="utf-8"))


def save_json(name, value):
    """결과를 output 폴더의 JSON 파일로 저장합니다."""
    (output_dir / name).write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")


# 지연 소포 3건: 소포 ID별 고객·상품 유형·상품·출발·도착 지역·약속 도착일
PARCELS = read_json("assignment02_parcels.json")
# 배송 추적 API가 돌려줄 값: 현재 위치·지연 일수·지연 사유
TRACKING = read_json("assignment02_tracking.json")
# 날씨 API가 돌려줄 값: 지역별 날씨·특보 여부
WEATHER = read_json("assignment02_weather.json")
# 보상 규정 API가 돌려줄 값: 상품 유형별 1일 보상액·최대 보상액·특보 감액률
POLICIES = read_json("assignment02_policies.json")


# 외부 API 흉내입니다. 수업용으로 일부러 실패하게 만든 함수입니다.
# CALLS에 API별 호출 횟수를 남깁니다. 실패 처리 장치가 몇 번 다시 불렀는지 이 숫자로 확인합니다.
CALLS = {"track": 0, "weather": 0, "policy": 0, "coupon": 0}


def track_parcel(parcel_id):
    """배송 추적 API: 현재 위치·지연 일수·지연 사유를 반환합니다. 세 번 중 앞의 두 번은 연결이 끊깁니다."""
    CALLS["track"] += 1
    # 일시 오류: 다시 부르면 성공합니다.
    if CALLS["track"] % 3 != 0:
        raise ConnectionError("배송 추적 API 연결이 끊겼습니다.")

    return TRACKING[parcel_id]


async def fetch_weather(region):
    """날씨 API: 지역의 날씨와 특보 여부를 반환합니다. 홀수 번째 호출은 20초 동안 응답하지 않습니다."""
    CALLS["weather"] += 1
    # 응답 없음: 오류 없이 기다리기만 합니다. 타임아웃이 없으면 20초를 그대로 기다립니다.
    if CALLS["weather"] % 2 == 1:
        await asyncio.sleep(20)

    return WEATHER[region]


def fetch_policy(kind):
    """보상 규정 API: 상품 유형의 1일 보상액·최대 보상액·특보 감액률을 반환합니다."""
    CALLS["policy"] += 1
    rule = POLICIES[kind]
    # 영구 오류: 냉장 규정은 응답 형식이 바뀌어 아래 키가 없습니다. 다시 불러도 같은 KeyError가 납니다.
    return {"1일 보상액": rule["1일 보상액"], "최대 보상액": rule["최대 보상액"], "특보 감액률": rule["특보 감액률"]}


# 쿠폰 발급 API 흉내입니다. 고객에게 쿠폰을 주는 부작용이 있는 수업용 함수입니다.
# 같은 멱등 키로 다시 부르면 새로 발급하지 않고 처음 쿠폰을 돌려줍니다(쿠폰 서버가 키로 중복을 막는 방식).
COUPONS = []
COUPON_KEYS = {}
# True로 두면 다음 발급 한 번은 쿠폰을 만든 뒤 응답이 끊깁니다(수업용 고장 스위치).
FAULTS = {"coupon_response_lost": False}


def issue_coupon(parcel_id, amount, key=""):
    """보상 쿠폰을 발급하고 쿠폰 dict(coupon_id·parcel_id·amount)를 반환합니다. 이미 쓴 key이면 다시 발급하지 않습니다."""
    CALLS["coupon"] += 1
    if key != "" and key in COUPON_KEYS:
        return COUPON_KEYS[key]

    coupon = {"coupon_id": f"CP-{len(COUPONS) + 1:04d}", "parcel_id": parcel_id, "amount": amount}
    COUPONS.append(coupon)
    if key != "":
        COUPON_KEYS[key] = coupon

    # 발급은 끝났는데 응답만 끊긴 상황입니다. 부른 쪽은 발급됐는지 알 수 없습니다.
    if FAULTS["coupon_response_lost"]:
        FAULTS["coupon_response_lost"] = False
        raise ConnectionError("쿠폰 API 응답이 끊겼습니다.")

    return coupon


def reset_calls():
    """호출 횟수, 발급한 쿠폰, 고장 스위치를 처음 상태로 되돌립니다."""
    for name in CALLS:
        CALLS[name] = 0
    COUPONS.clear()
    COUPON_KEYS.clear()
    FAULTS["coupon_response_lost"] = False


# 오늘 처리할 지연 소포 3건을 확인합니다.
for parcel_id in PARCELS:
    row = PARCELS[parcel_id]
    print(parcel_id, row["상품"], f"({row['상품 유형']})", "|", row["출발"], "->", row["도착 지역"], "| 약속 도착일:", row["약속 도착일"])

## 1. 일시 오류만 다시 시도하는 정책을 만드세요

다시 시도해서 나아지는 오류만 노드에서 다시 시도해야 합니다. 응답 형식이 바뀐 것 같은 영구 오류를 다시 시도하면 시간과 호출만 늘어납니다. 아래 계층이 이미 다시 시도하는 오류도 노드에서 또 시도하지 않습니다. 모델 호출은 OpenAI SDK가 시도 3번까지 보내므로, 노드도 다시 시도하면 시도 수가 곱해집니다(교안 02 5-3의 겹친 재시도). `RetryPolicy`의 `retry_on`에 예외를 받아 `True`/`False`를 돌려주는 함수를 넣으면 재시도 여부를 직접 정할 수 있습니다. `is_transient`는 노드 코드에서도 같은 기준으로 부를 수 있게 예외 클래스 튜플 대신 함수로 만듭니다.

| 예외 | 이 과제에서 나오는 곳 | 노드에서 다시 시도 |
|---|---|---|
| `ConnectionError` | 배송 추적 API 연결 끊김, 쿠폰 API 응답 끊김 | 예 |
| `NodeTimeoutError` | 노드 타임아웃(날씨 API 응답 없음). 내장 `TimeoutError`를 상속하지 않습니다. | 예 |
| `TimeoutError` | 이 과제에는 없음(네트워크 라이브러리의 시간 초과) | 예 |
| `openai.APIConnectionError` 등 OpenAI SDK 오류 | 모델 호출(`writer`). 내장 `ConnectionError`를 상속하지 않습니다. | 아니오(SDK가 이미 다시 시도) |
| `KeyError`, `ValueError` 등 그 밖의 예외 | 냉장 규정 응답 형식 변경 | 아니오 |

**작성:** `is_transient(exc)` 함수와 재시도 정책 `RETRY`를 만드세요.

**목적:** 조회 워커가 일시 오류일 때만 다시 실행되게 합니다.

**구현할 처리**
1. `exc`가 표에서 '예'인 세 가지 중 하나이면 `True`, 아니면 `False`를 반환합니다.
2. `RetryPolicy`로 `RETRY`를 만듭니다. 첫 시도를 포함해 최대 3번(`max_attempts`), 첫 재시도 전 0.5초 대기(`initial_interval`), 재시도 여부는 `is_transient` 함수(`retry_on`)로 정합니다.

**반환값:** `is_transient`는 `True` 또는 `False`를 반환합니다. `RETRY`는 아래 값을 가진 `RetryPolicy`입니다.

| 필드 | 값 |
|---|---|
| `max_attempts` | `3` |
| `initial_interval` | `0.5` |
| `retry_on` | `is_transient` 함수 |

**반환 예시**

```python
is_transient(ConnectionError("연결 끊김"))   # True
is_transient(KeyError("1일 보상액"))        # False
is_transient(openai.APIConnectionError(request=...))  # False
```

In [ ]:
# [작성] 일시 오류인지 가르는 함수와 재시도 정책을 만드세요.
def is_transient(exc):
    """노드에서 다시 시도할 일시 오류이면 True를 반환합니다."""
    # 1. 표에서 '예'인 세 예외 중 하나이면 True, 아니면 False를 반환하세요.
    ...


# 2. 첫 시도 포함 3번, 첫 재시도 전 0.5초 대기, is_transient로 재시도 여부를 정하는 RETRY를 만드세요.
...

In [ ]:
# [자가채점] 예외 종류별 판정과 재시도 정책 값을 확인합니다.
import httpx

model_error = openai.APIConnectionError(request=httpx.Request("POST", "https://api.openai.com/v1/responses"))
cases = [
    (ConnectionError("연결 끊김"), True),
    (NodeTimeoutError("check_worker", 2.1, kind="run", run_timeout=2.0), True),
    (TimeoutError("시간 초과"), True),
    (KeyError("1일 보상액"), False),
    (ValueError("형식 오류"), False),
    (RuntimeError("알 수 없는 오류"), False),
    (model_error, False),
]
for exc, expected in cases:
    result = is_transient(exc)
    assert result is expected, f"{type(exc).__name__}: is_transient는 {expected}를 반환해야 합니다. 표에서 '예'인 세 예외만 True입니다. 지금: {result}"
assert isinstance(RETRY, RetryPolicy), "RETRY는 RetryPolicy로 만드세요."
assert RETRY.max_attempts == 3, "max_attempts는 첫 시도를 포함한 횟수입니다. 3으로 두세요."
assert RETRY.initial_interval == 0.5, "첫 재시도 전 대기(initial_interval)는 0.5초입니다."
assert RETRY.retry_on is is_transient, "retry_on에는 is_transient 함수 자체를 넣으세요(호출한 결과가 아니라 함수)."
print("재시도 정책 확인 완료")

## 2. 조회 워커 하나의 실패가 팀 전체를 멈추지 않게 하세요

`Send`로 펼친 조회 워커 셋은 같은 단계에서 동시에 실행됩니다. 한 워커가 예외를 내면 그 단계 전체가 실패하고, 다른 워커가 얻은 결과도 다음 노드로 넘어가지 못합니다. 그래서 워커 안에서 실패를 둘로 나눕니다.

- 일시 오류: 예외를 그대로 올립니다. `RetryPolicy`가 그 워커만 다시 실행합니다. 3번 모두 실패하면 그 단계 전체가 실패합니다.
- 그 밖의 실패: 예외를 올리지 않고 `failures`에 기록한 뒤 정상 반환합니다. 나머지 워커의 결과로 팀이 이어 갑니다.

이 버전(langgraph 1.2.9)에서는 `Send`로 펼친 노드에 `error_handler`를 붙여도 같은 단계의 실패가 다시 올라오므로, 부분 실패는 워커 안에서 처리합니다. 워커 셋이 같은 단계에 `facts`와 `failures`를 함께 쓰므로 두 필드에는 dict를 합치는 리듀서가 필요합니다.

<img src="images/node_focus/assignment02_check_worker.png" width="1050" alt="팀 그래프에서 Send로 펼쳐지는 check_worker 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `CHECK_ITEMS` | 조회 항목 `["track", "weather", "policy"]` |
| `CheckTask` | Send로 워커에 넘기는 작업. `item`(조회 항목), `parcel_id`(소포 ID) |
| `call_api(item, parcel_id)` | async 함수. 항목에 맞는 외부 API를 불러 결과를 반환합니다. API가 낸 예외는 그대로 올라옵니다. |

**작성:** 리듀서 `merge_dicts(old, new)`와 async 노드 `check_worker(task)`를 완성하세요.

**목적:** 영구 오류 하나가 생겨도 나머지 조회 결과로 보상 처리를 이어 가고, 일시 오류는 재시도에 맡깁니다.

**구현할 처리**
1. `merge_dicts`: `old`를 바꾸지 말고 복사한 dict에 `new`를 합쳐 반환합니다. 같은 키는 `new`의 값을 씁니다.
2. `check_worker`: `call_api`로 `task`의 항목을 조회합니다(`await`).
3. 예외가 나면 `is_transient`로 가릅니다. 일시 오류이면 그대로 다시 올리고, 아니면 `failures`에 `{항목: "예외 이름: 예외 내용"}`을 담아 반환합니다. 예외 이름은 예외 클래스의 이름입니다.
4. 성공하면 `facts`에 `{항목: 조회 결과}`를 담아 반환합니다.

**반환값**

| 함수 | 반환할 값 |
|---|---|
| `merge_dicts` | `old`와 `new`의 항목을 모두 가진 새 dict |
| `check_worker` 성공 | `{"facts": {항목: 조회 결과}}` |
| `check_worker` 영구 오류 | `{"failures": {항목: "예외 이름: 예외 내용"}}` |
| `check_worker` 일시 오류 | 반환하지 않고 예외를 다시 올림 |

**반환 예시:** 냉장 소포 P-7002의 규정 조회와, P-7001의 규정 조회입니다.

```python
{"failures": {"policy": "KeyError: '1일 보상액'"}}
{"facts": {"policy": {"1일 보상액": 1000, "최대 보상액": 5000, "특보 감액률": 50}}}
```

In [ ]:
# 조회 항목과 Send로 넘길 작업 모양, 항목별로 외부 API를 고르는 함수입니다.
CHECK_ITEMS = ["track", "weather", "policy"]


class CheckTask(TypedDict):
    item: str  # 조회 항목: "track", "weather", "policy"
    parcel_id: str  # 소포 ID


async def call_api(item, parcel_id):
    """조회 항목에 맞는 외부 API를 불러 결과를 반환합니다. API가 낸 예외는 그대로 올라갑니다."""
    parcel = PARCELS[parcel_id]
    if item == "track":
        return track_parcel(parcel_id)

    # 날씨 API만 async라 await로 기다립니다.
    if item == "weather":
        return await fetch_weather(parcel["도착 지역"])

    return fetch_policy(parcel["상품 유형"])

In [ ]:
# [작성] 두 dict를 합치는 리듀서와, 실패를 격리하는 조회 워커를 만드세요.
def merge_dicts(old, new):
    """기존 dict에 새 dict를 합친 새 dict를 반환합니다. 같은 키는 새 값을 씁니다."""
    # 1. 입력을 바꾸지 말고 old의 복사본에 new를 합쳐 반환하세요.
    ...


async def check_worker(task: CheckTask) -> dict:
    """조회 항목 하나를 외부 API로 확인하고, 성공은 facts에, 일시 오류가 아닌 실패는 failures에 기록합니다."""
    # 2. call_api로 task의 항목을 조회하세요. 예외가 나면 3번을 합니다.
    # 3. 일시 오류이면 그대로 다시 올리고, 아니면 failures에 {항목: "예외 이름: 예외 내용"}을 담아 반환하세요.
    ...

    # 4. 성공하면 facts에 {항목: 조회 결과}를 담아 반환하세요.
    ...

In [ ]:
# [자가채점] 리듀서와 워커를 직접 불러 확인합니다. 외부 API 흉내만 부르고 모델은 부르지 않습니다.
old = {"track": 1}
merged = merge_dicts(old, {"weather": 2})
assert merged == {"track": 1, "weather": 2}, f"두 dict의 항목을 모두 담아 반환하세요. 지금: {merged}"
assert old == {"track": 1}, "입력 dict를 바꾸지 말고 복사본에 합치세요."
assert merge_dicts({"policy": 1}, {"policy": 2}) == {"policy": 2}, "같은 키는 새 값을 쓰세요."

reset_calls()
result = await check_worker({"item": "policy", "parcel_id": "P-7002"})
assert result == {"failures": {"policy": "KeyError: '1일 보상액'"}}, f"영구 오류는 failures에 {{항목: '예외 이름: 예외 내용'}}으로 담아 반환하세요. 지금: {result}"
result = await check_worker({"item": "policy", "parcel_id": "P-7001"})
assert result == {"facts": {"policy": {"1일 보상액": 1000, "최대 보상액": 5000, "특보 감액률": 50}}}, f"성공하면 facts에 {{항목: 조회 결과}}를 담아 반환하세요. 지금: {result}"

# 배송 추적 API의 첫 호출은 연결이 끊깁니다.
reset_calls()
raised = False
try:
    await check_worker({"item": "track", "parcel_id": "P-7001"})
except ConnectionError:
    raised = True
assert raised, "일시 오류(ConnectionError)는 failures에 기록하지 말고 다시 올리세요. 그래야 RetryPolicy가 재시도합니다."

# 날씨 API는 짝수 번째 호출에 바로 응답합니다.
CALLS["weather"] = 1
result = await check_worker({"item": "weather", "parcel_id": "P-7003"})
assert result == {"facts": {"weather": {"날씨": "태풍", "특보": True}}}, f"날씨 조회 결과를 facts에 담으세요. 지금: {result}"
reset_calls()
print("조회 워커 확인 완료")

## 3. 조회가 실패한 뒤 계속 처리할지 판단하세요

2번처럼 영구 오류를 `failures`에 기록하면 팀은 무엇이 실패하든 다음 단계로 갑니다. 보상 규정이 빠진 소포는 기본 보상액으로 처리해도 되지만, **배송 추적**이 빠진 소포는 지연 일수를 몰라 보상액을 정할 수 없습니다. 교안 03 2-7처럼 조회를 필수 조회와 선택 조회로 나누고, 무엇이 실패했는지로 이번 실행의 결과(`outcome`)를 정하는 **진행 판단** 노드를 만듭니다. 같은 실패에는 늘 같은 결론이 나야 하므로 LLM 없이 코드로 판단합니다.

| 조회 | 구분 | 이유 |
|---|---|---|
| `track` (배송 추적) | 필수 조회 | 지연 일수를 모르면 보상액을 정할 수 없습니다 |
| `weather` (날씨), `policy` (보상 규정) | 선택 조회 | 빠지면 기본 보상액과 담당자 확인 안내로 처리합니다 |

| 조회 결과 | `outcome` | 다음 노드(`goto`) | `outcome_reason`에 담을 것 |
|---|---|---|---|
| 실패한 조회 없음 | `"completed"` | `"quote"` | 판단 이유(빈 문자열이 아닌 문장) |
| 모든 조회 실패 | `"failed"` | `END` | 실패한 항목 이름 모두 |
| 필수 조회 실패(모두 실패한 경우는 위 줄) | `"held"` | `"escalate"` | 실패한 필수 항목 이름 |
| 그 밖(선택 조회만 실패) | `"degraded"` | `"quote"` | 실패한 항목 이름 모두 |

표의 위에서부터 차례로 확인합니다. 이유에는 이번에 실패한 항목 이름만 넣습니다.

`held`는 쿠폰을 보내지 않고 사람 확인 요청 노드(`escalate`)로 보내 담당자가 넘겨받게 합니다. `held`와 `failed`는 예외가 아니라 정상 종료라서, 실행이 끝난 뒤 `outcome`을 봐야 알 수 있습니다. 판단 노드는 조회 워커가 모두 끝난 뒤 한 번 실행되고, 다음 노드와 판단 결과를 `Command` 하나로 반환합니다. 그래서 Langfuse에서 이 노드를 누르면 결론과 이유가 함께 보입니다.

<img src="images/node_focus/assignment02_decide_after_checks.png" width="1050" alt="팀 그래프에서 decide_after_checks 노드만 강조합니다. 이 노드에서 quote, escalate, END로 가는 갈래가 있습니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `TeamState` | 팀의 공유 데이터. `facts`, `failures`는 2번의 `merge_dicts`로 합치고, 판단 결과는 `outcome`, `outcome_reason`에 남깁니다. |
| `CHECK_ITEMS` | 조회 항목 `["track", "weather", "policy"]` (2번) |
| `REQUIRED_ITEMS` | 필수 조회 목록 `["track"]`. 여기에 없는 조회 항목은 선택 조회입니다. |

**작성:** 판단 노드 `decide_after_checks(state)`를 완성하세요.

**목적:** 필수 조회가 빠진 소포에 근거 없는 보상을 보내지 않고, 선택 조회만 빠진 소포는 이어서 처리합니다.

**구현할 처리**
1. `state["failures"]`가 비었으면 `completed`로 `quote`에 보냅니다.
2. `CHECK_ITEMS`의 조회가 모두 실패했으면 `failed`로 끝냅니다(`END`).
3. `REQUIRED_ITEMS`에 든 항목이 실패했으면 `held`로 `escalate`에 보냅니다. 항목 이름을 코드에 직접 적지 말고 `REQUIRED_ITEMS`에서 읽습니다.
4. 그 밖(선택 조회만 실패)은 `degraded`로 `quote`에 보냅니다.

어느 경우든 `Command(goto=다음 노드, update={"outcome": ..., "outcome_reason": ...})`를 반환합니다.

**반환값: `Command`**

| 키 | 반환할 값 |
|---|---|
| `goto` | 표의 다음 노드 |
| `update["outcome"]` | `"completed"`, `"degraded"`, `"held"`, `"failed"` 중 하나 |
| `update["outcome_reason"]` | 판단 이유 문자열. 실패가 있으면 표대로 항목 이름(`track`처럼)을 넣습니다 |

**반환 예시:** 보상 규정 조회만 실패한 경우와, 배송 추적 조회가 실패한 경우입니다.

```python
Command(goto="quote", update={"outcome": "degraded", "outcome_reason": "선택 조회 실패: policy, 확인한 사실로 진행"})
Command(goto="escalate", update={"outcome": "held", "outcome_reason": "필수 조회 실패: track, 보상 보류 후 담당자 확인"})
```

In [ ]:
# 팀 State와 필수 조회 목록입니다.
class TeamState(TypedDict):
    parcel_id: str  # 소포 ID
    parcel: str  # 소포 정보 문자열(고객·상품·구간·약속 도착일)
    facts: Annotated[dict, merge_dicts]  # 조회 워커: {항목: 결과}. 같은 단계의 워커 셋이 함께 쓰므로 리듀서로 합칩니다
    failures: Annotated[dict, merge_dicts]  # 실패 기록: {항목 또는 노드: "예외 이름: 예외 내용"}
    outcome: str  # 진행 판단: completed, degraded, held, failed 중 하나
    outcome_reason: str  # 진행 판단 이유
    amount: int  # 보상액(원)
    manual_check: bool  # 담당자 확인이 필요한지
    draft: str  # 고객 안내문 초안
    rejections: list[str]  # 검수 반려 사유 기록(오래된 것부터)
    approved: bool  # 검수 통과 여부
    status: str  # 처리 상태: "보상 완료" 또는 "사람 확인 필요"
    coupon_id: str  # 발급한 쿠폰 ID


# 빠지면 보상액을 정할 수 없는 필수 조회입니다. 나머지(weather, policy)는 선택 조회입니다.
REQUIRED_ITEMS = ["track"]

In [ ]:
# [작성] 조회 실패 기록으로 이번 소포를 계속 처리할지 정하는 판단 노드를 만드세요.
def decide_after_checks(state: TeamState) -> Command[Literal["quote", "escalate", "__end__"]]:
    """실패한 조회로 completed, degraded, held, failed 중 하나를 정하고 다음 노드로 보냅니다."""
    # 1. 실패한 조회가 없으면 completed로 quote에 보내세요.
    ...

    # 2. 모든 조회가 실패했으면 failed로 끝내세요.
    ...

    # 3. REQUIRED_ITEMS의 조회가 실패했으면 held로 escalate에 보내세요.
    ...

    # 4. 선택 조회만 실패했으면 degraded로 quote에 보내세요.
    ...

In [ ]:
# [자가채점] 조회 결과 예시마다 판단 결과와 다음 노드를 확인합니다. 모델은 부르지 않습니다.
SAMPLE_FACTS = {"track": TRACKING["P-7001"], "weather": WEATHER["부산"], "policy": POLICIES["일반"]}


def checked_state(failed_items):
    """failed_items는 failures에, 나머지 조회 항목은 facts에 담은 State를 반환합니다."""
    facts = {}
    failures = {}
    for item in CHECK_ITEMS:
        if item in failed_items:
            failures[item] = "KeyError: '응답 형식 변경'"
        else:
            facts[item] = SAMPLE_FACTS[item]
    return {"parcel_id": "P-7001", "facts": facts, "failures": failures}


cases = [
    ([], "completed", "quote"),
    (["policy"], "degraded", "quote"),
    (["weather"], "degraded", "quote"),
    (["weather", "policy"], "degraded", "quote"),
    (["track"], "held", "escalate"),
    (["track", "policy"], "held", "escalate"),
    (["track", "weather", "policy"], "failed", END),
]
for failed_items, outcome, goto in cases:
    step = decide_after_checks(checked_state(failed_items))
    assert isinstance(step, Command), "Command(goto=..., update=...)를 반환하세요. `...`를 채웠는지도 확인하세요."
    assert isinstance(step.update, dict) and sorted(step.update) == ["outcome", "outcome_reason"], f"update에는 outcome과 outcome_reason 두 키만 담으세요. 지금: {step.update}"
    got = step.update["outcome"]
    if failed_items == ["track", "weather", "policy"]:
        assert got == "failed", f"모든 조회가 실패하면 failed입니다. 모두 실패하면 필수 조회도 실패한 것이니, 모두 실패했는지를 먼저 보세요. 지금: {got}"
    assert got == outcome, f"실패한 조회가 {failed_items}일 때 outcome이 표와 다릅니다. 필수 조회(REQUIRED_ITEMS)와 선택 조회를 나눠 보세요. 지금: {got}"
    assert step.goto == goto, f"outcome이 {outcome}이면 goto는 표의 다음 노드입니다. 지금: {step.goto}"
    reason = step.update["outcome_reason"]
    assert isinstance(reason, str) and reason.strip() != "", "outcome_reason에 판단 이유 문자열을 담으세요."
    named = failed_items
    if outcome == "held":
        named = ["track"]
    for item in named:
        assert item in reason, f"outcome_reason에 실패한 항목 이름 {item}을 넣으세요. 지금: {reason}"
    for item in ["track", "weather", "policy"]:
        if item not in failed_items:
            assert item not in reason, f"실패하지 않은 조회 {item}이 outcome_reason에 들어 있습니다. 이유는 이번 실패 기록(failures)에서 만드세요. 지금: {reason}"

# 필수 조회를 REQUIRED_ITEMS에서 읽는지 확인합니다. 바꾼 목록은 끝나면 되돌립니다.
REQUIRED_ITEMS[:] = ["policy"]
try:
    swapped = decide_after_checks(checked_state(["policy"]))
    kept = decide_after_checks(checked_state(["track"]))
finally:
    REQUIRED_ITEMS[:] = ["track"]
assert swapped.update["outcome"] == "held" and kept.update["outcome"] == "degraded", "필수 조회는 REQUIRED_ITEMS에서 읽으세요. 항목 이름을 코드에 직접 적으면 목록이 바뀔 때 판단이 틀립니다."
assert "policy" in swapped.update["outcome_reason"], f"held의 outcome_reason에는 실패한 필수 항목 이름을 넣으세요. 지금: {swapped.update['outcome_reason']}"
print("진행 판단 확인 완료")

## 4. 보상액 계산이 실패하면 기본 보상액으로 이어 가게 하세요

3번 판단에서 선택 조회만 빠진 소포는 `quote`로 갑니다. 규정 조회가 실패한 소포는 `facts`에 `policy`가 없어 `quote`에서 `KeyError`가 납니다. 교안 03의 3절처럼 `error_handler`로 붙인 핸들러는 노드가 재시도까지 모두 실패했을 때 대신 실행되고, 실패 정보 `NodeError`(`node`: 실패한 노드 이름, `error`: 예외)를 받습니다. 원래 노드의 고정 엣지는 쓰이지 않으므로 다음 노드를 `Command(goto=...)`로 정합니다.

<img src="images/node_focus/assignment02_quote.png" width="1050" alt="팀 그래프에서 quote 노드만 강조합니다. quote에서 writer로 가는 error_handler 폴백 선이 있습니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `quote(state)` | 규정·지연 일수·특보로 `{"amount": 보상액, "manual_check": False}`를 반환합니다. 규정이 없으면 `KeyError`가 납니다. |
| `DEFAULT_AMOUNT` | 규정을 읽지 못했을 때 먼저 지급하는 기본 보상액 `3000`(원) |

**작성:** `quote_error_handler(state, error)` 핸들러를 완성하세요.

**목적:** 규정 API가 고장 나도 고객에게 기본 보상과 담당자 확인 안내를 보내고, 실패를 기록에 남깁니다.

**구현할 처리**
1. `error.error`의 예외로 `"예외 이름: 예외 내용"` 문자열을 만듭니다.
2. `writer`로 가는 `Command`를 반환합니다. `amount`는 `DEFAULT_AMOUNT`, `manual_check`는 `True`, `failures`는 `{error.node: 1에서 만든 문자열}`로 갱신합니다.

**반환값: `Command`**

| 키 | 반환할 값 |
|---|---|
| `goto` | `"writer"` |
| `update["amount"]` | `DEFAULT_AMOUNT` |
| `update["manual_check"]` | `True` |
| `update["failures"]` | `{실패한 노드 이름: "예외 이름: 예외 내용"}` (리듀서가 기존 실패 기록에 합칩니다) |

**반환 예시:** `quote`가 `KeyError('policy')`로 실패한 경우입니다.

```python
Command(goto="writer", update={"amount": 3000, "manual_check": True,
                               "failures": {"quote": "KeyError: 'policy'"}})
```

In [ ]:
# 보상액 계산 노드와, 규정을 읽지 못했을 때 먼저 지급하는 기본 보상액입니다.
DEFAULT_AMOUNT = 3000


def quote(state: TeamState) -> dict:
    """규정·지연 일수·특보로 보상액을 계산합니다. 규정 조회가 실패했으면 facts에 policy가 없어 KeyError가 납니다."""
    policy = state["facts"]["policy"]
    days = state["facts"]["track"]["지연 일수"]
    amount = min(policy["1일 보상액"] * days, policy["최대 보상액"])

    # 기상 특보로 늦어진 경우 규정의 감액률만큼 줄입니다.
    if state["facts"]["weather"]["특보"]:
        amount = amount * (100 - policy["특보 감액률"]) // 100

    return {"amount": amount, "manual_check": False}

In [ ]:
# [작성] 보상액 계산이 실패했을 때 기본 보상액으로 writer에 보내는 핸들러를 만드세요.
def quote_error_handler(state: TeamState, error: NodeError) -> Command[Literal["writer"]]:
    """보상액 계산이 실패하면 기본 보상액과 담당자 확인 표시를 남기고 writer로 보냅니다."""
    # 1. error의 예외로 "예외 이름: 예외 내용" 문자열을 만드세요.
    ...

    # 2. writer로 가면서 amount·manual_check·failures를 갱신하는 Command를 반환하세요.
    ...

In [ ]:
# [자가채점] 핸들러를 직접 불러 반환값을 확인합니다.
state = {"parcel_id": "P-7002", "facts": {"track": {"지연 일수": 1}}, "failures": {"policy": "KeyError: '1일 보상액'"}}
raised = False
try:
    quote(state)
except KeyError:
    raised = True
assert raised, "규정 조회가 실패한 소포는 quote에서 KeyError가 나야 합니다. 제공 코드를 바꾸지 마세요."

step = quote_error_handler(state, NodeError(node="quote", error=KeyError("policy")))
assert isinstance(step, Command), "Command(goto=..., update=...)를 반환하세요. dict만 반환하면 핸들러 뒤 다음 노드가 정해지지 않아 그래프가 그 자리에서 끝납니다."
assert step.goto == "writer", "핸들러가 끝나면 writer로 가야 고객 안내가 이어집니다."
expected = {"amount": DEFAULT_AMOUNT, "manual_check": True, "failures": {"quote": "KeyError: 'policy'"}}
assert step.update == expected, f"update에는 amount(DEFAULT_AMOUNT), manual_check(True), failures({{노드 이름: '예외 이름: 예외 내용'}})를 담으세요. 지금: {step.update}"
other = quote_error_handler(state, NodeError(node="quote", error=ValueError("규정 형식 오류")))
assert other.update["failures"] == {"quote": "ValueError: 규정 형식 오류"}, f"failures 값은 받은 error.error로 만드세요. 지금: {other.update['failures']}"
print("폴백 핸들러 확인 완료")

## 5. 같은 반려가 반복되면 사람에게 넘기세요

안내문 작성(`writer`, 모델)과 검수(`reviewer`, 규칙 코드)는 반려되면 다시 쓰는 사이클입니다. 모델이 같은 실수를 고치지 못하면 두 노드가 같은 사유로 끝없이 오가는 핑퐁이 됩니다. 예외 없이 모델 호출만 늘다가 `recursion_limit`에 닿아야 멈춥니다. 같은 사유가 연속으로 반복되면 고칠 수 없다는 신호로 보고 사람 확인으로 넘깁니다.

교안 02 3-2의 가드는 앞에 나온 사유가 다시 나오면 바로 넘겼지만, 이 과제는 **연속 반복**만 셉니다. A, B, A처럼 떨어져서 다시 나온 사유에는 고칠 기회를 한 번 더 주고, 반려 수 상한(`MAX_REVIEWS`)이 끝을 보장합니다.

<img src="images/node_focus/assignment02_reviewer.png" width="1050" alt="팀 그래프에서 reviewer 노드만 강조합니다. reviewer 뒤에는 writer, escalate, issue로 가는 갈래가 있습니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `writer(state)` | 사실·보상액·지난 반려 사유로 안내문을 써서 `{"draft": 초안}`을 반환합니다(모델 호출). |
| `reviewer(state)` | 규칙 검사를 통과하면 `{"approved": True}`, 아니면 `{"approved": False, "rejections": 기존 사유 + [새 사유]}`를 반환합니다. |
| `PING_PONG_LIMIT` | 같은 사유가 연속으로 이만큼 반복되면 핑퐁으로 봅니다: `2` |
| `MAX_REVIEWS` | 사유가 달라도 반려가 이만큼 쌓이면 사람에게 넘깁니다: `4` |

**작성:** `is_ping_pong(rejections, limit)`와 `route_after_review(state)`를 완성하세요.

**목적:** 같은 사유로 반려가 되풀이되면 사람 확인(`escalate`)으로 넘깁니다.

**구현할 처리**
1. `is_ping_pong`: 반려 사유가 `limit`개보다 적으면 `False`를 반환합니다.
2. `is_ping_pong`: 마지막 `limit`개의 사유가 모두 같으면 `True`, 아니면 `False`를 반환합니다. 떨어져서 두 번 나온 사유는 연속이 아닙니다.
3. `route_after_review`: `state["approved"]`가 참이면 `"issue"`를 반환합니다.
4. `route_after_review`: 같은 사유가 `PING_PONG_LIMIT`번 연속이거나 반려가 `MAX_REVIEWS`번 이상이면 `"escalate"`를 반환합니다.
5. `route_after_review`: 아니면 `"writer"`를 반환합니다.

**반환값**

| 함수 | 반환할 값 |
|---|---|
| `is_ping_pong` | `True` 또는 `False` |
| `route_after_review` | `"issue"`, `"escalate"`, `"writer"` 중 하나 |

**반환 예시**

```python
is_ping_pong(["소포 ID가 없습니다.", "보상액 2,000원이 없습니다.", "보상액 2,000원이 없습니다."], 2)  # True
is_ping_pong(["보상액 2,000원이 없습니다.", "소포 ID가 없습니다.", "보상액 2,000원이 없습니다."], 2)  # False
```

In [ ]:
# 안내문 작성(모델)과 검수(규칙 코드) 노드입니다. 두 노드는 반려되면 다시 쓰는 사이클을 이룹니다.
writer_prompt = ChatPromptTemplate.from_messages([
    ("system", "바로택배 고객센터의 배송 지연 안내문 작성 담당입니다. 확인한 사실과 보상액으로 고객에게 보낼 한국어 안내문을 "
     "300자 이내로 쓰세요. 소포 ID와 보상액을 주어진 표기 그대로(예: 2,000원) 넣고, 보상은 쿠폰으로 드린다고 쓰세요. "
     "확인하지 못한 항목이 있으면 확인하지 못했다고 쓰고, 담당자 확인 필요가 '예'이면 담당자가 다시 확인해 연락드린다는 문장을 넣으세요. "
     "지난 반려 사유가 있으면 그 사유를 고치세요."),
    ("human", "소포 ID: {parcel_id}\n소포 정보:\n{parcel}\n\n확인한 사실:\n{facts}\n\n확인하지 못한 항목: {failures}\n\n"
     "보상액: {amount_text}\n담당자 확인 필요: {manual_check}\n지난 반려 사유: {rejection}"),
])
writer_chain = writer_prompt | llm


def writer(state: TeamState) -> dict:
    """확인한 사실과 보상액, 지난 반려 사유로 고객 안내문 초안을 씁니다."""
    # 바로 앞 반려 사유만 넘겨 그 사유부터 고치게 합니다.
    rejection = "없음"
    if state["rejections"]:
        rejection = state["rejections"][-1]
    manual = "아니오"
    if state["manual_check"]:
        manual = "예"
    failures = "없음"
    if state["failures"]:
        failures = json.dumps(state["failures"], ensure_ascii=False)

    draft = writer_chain.invoke({
        "parcel_id": state["parcel_id"],
        "parcel": state["parcel"],
        "facts": json.dumps(state["facts"], ensure_ascii=False),
        "failures": failures,
        "amount_text": f"{state['amount']:,}원",
        "manual_check": manual,
        "rejection": rejection,
    }).text
    return {"draft": draft}


# 첫 위반 하나만 사유로 돌려줍니다. 검사 순서가 같아야 같은 실수가 같은 사유로 기록되어 핑퐁 판정이 됩니다.
def review_draft(state):
    """안내문 초안이 지켜야 할 규칙을 검사해 첫 위반 사유를 반환합니다. 통과하면 빈 문자열입니다."""
    draft = state["draft"]
    amount_text = f"{state['amount']:,}원"
    if state["parcel_id"] not in draft:
        return "소포 ID가 없습니다."
    if amount_text not in draft:
        return f"보상액 {amount_text}이 없습니다."
    if len(draft) > 300:
        return "300자를 넘습니다."
    if state["manual_check"] and "담당자" not in draft:
        return "담당자 확인 안내가 없습니다."
    return ""


def reviewer(state: TeamState) -> dict:
    """규칙 검사로 통과 여부를 정하고, 반려하면 사유를 기록에 덧붙입니다."""
    reason = review_draft(state)
    if reason == "":
        return {"approved": True}

    return {"approved": False, "rejections": state["rejections"] + [reason]}


# 같은 사유가 연속으로 이만큼 반복되면 핑퐁으로 봅니다.
PING_PONG_LIMIT = 2
# 사유가 달라도 반려가 이만큼 쌓이면 사람에게 넘깁니다.
MAX_REVIEWS = 4

In [ ]:
# [작성] 같은 반려 사유의 반복을 감지하고, 검수 뒤 다음 노드를 정하세요.
def is_ping_pong(rejections, limit):
    """마지막 limit개의 반려 사유가 모두 같으면 True를 반환합니다."""
    # 1. 반려가 limit개보다 적으면 False를 반환하세요.
    ...

    # 2. 마지막 limit개가 모두 같은 사유이면 True, 아니면 False를 반환하세요.
    ...


def route_after_review(state: TeamState) -> Literal["writer", "escalate", "issue"]:
    """검수 결과와 반려 기록으로 다음 노드 이름을 정합니다."""
    # 3. 통과했으면 "issue"를 반환하세요.
    ...

    # 4. 같은 사유가 PING_PONG_LIMIT번 연속이거나 반려가 MAX_REVIEWS번 이상이면 "escalate"를 반환하세요.
    ...

    # 5. 아니면 다시 쓰도록 "writer"를 반환하세요.
    ...

In [ ]:
# [자가채점] 반려 기록 예시로 감지 함수와 분기를 확인합니다.
cases = [
    ([], 2, False),
    (["보상액 2,000원이 없습니다."], 2, False),
    (["보상액 2,000원이 없습니다.", "보상액 2,000원이 없습니다."], 2, True),
    (["소포 ID가 없습니다.", "보상액 2,000원이 없습니다.", "보상액 2,000원이 없습니다."], 2, True),
    (["보상액 2,000원이 없습니다.", "소포 ID가 없습니다.", "보상액 2,000원이 없습니다."], 2, False),
    (["300자를 넘습니다.", "300자를 넘습니다."], 3, False),
    (["300자를 넘습니다.", "300자를 넘습니다.", "300자를 넘습니다."], 3, True),
]
for rejections, limit, expected in cases:
    result = is_ping_pong(rejections, limit)
    assert result is expected, f"is_ping_pong({rejections}, {limit})는 {expected}여야 합니다. 마지막 limit개가 모두 같은지만 보세요. 지금: {result}"

same = ["보상액 2,000원이 없습니다.", "보상액 2,000원이 없습니다."]
mixed = ["소포 ID가 없습니다.", "300자를 넘습니다.", "소포 ID가 없습니다.", "300자를 넘습니다."]
assert route_after_review({"approved": True, "rejections": same}) == "issue", "통과했으면 반려 기록과 상관없이 issue로 보내세요."
assert route_after_review({"approved": False, "rejections": same}) == "escalate", "같은 사유가 PING_PONG_LIMIT번 연속이면 escalate로 보내세요."
assert route_after_review({"approved": False, "rejections": mixed}) == "escalate", "사유가 달라도 반려가 MAX_REVIEWS번 이상이면 escalate로 보내세요."
assert route_after_review({"approved": False, "rejections": mixed[:2]}) == "writer", "반복이 아니고 상한 전이면 writer로 보내 다시 쓰게 하세요."
print("핑퐁 가드 확인 완료")

## 6. 쿠폰이 두 번 발급되지 않게 하세요

쿠폰 발급처럼 바깥에 흔적을 남기는 호출은 응답이 끊겼을 때가 위험합니다. 쿠폰은 이미 발급됐는데 부른 쪽은 실패로 보고 다시 실행하면 쿠폰이 두 장 나갑니다. **멱등 키**는 같은 요청에 늘 같은 값을 붙여, 받는 쪽이 이미 처리한 요청을 알아보게 하는 값입니다. `issue_coupon`은 이미 쓴 키가 오면 새로 발급하지 않고 처음 쿠폰을 돌려줍니다.

<img src="images/node_focus/assignment02_issue.png" width="1050" alt="팀 그래프에서 issue 노드만 강조합니다.">

| 제공 코드 | 입력과 반환값 |
|---|---|
| `issue_coupon(parcel_id, amount, key)` | 쿠폰 `{"coupon_id", "parcel_id", "amount"}`를 반환합니다. 이미 쓴 `key`이면 새로 발급하지 않습니다. 응답이 끊기면 `ConnectionError`를 냅니다. |
| `COUPONS`, `COUPON_KEYS` | 발급된 쿠폰 목록과 키별 쿠폰. 채점이 확인합니다. |
| `escalate(state)` | 사람 확인 요청 노드. `{"status": "사람 확인 필요"}`를 반환합니다. 3번의 `held`도 이 노드로 옵니다. |

**작성:** `issue(state)` 노드를 완성하세요.

**목적:** 같은 소포는 몇 번 다시 실행해도 쿠폰이 한 장만 발급되게 합니다.

**구현할 처리**
1. 소포 ID로 멱등 키를 만듭니다. 모양은 `"P-7001:지연보상"`입니다(소포 ID 뒤에 `:지연보상`).
2. `issue_coupon`에 소포 ID, 보상액(`state["amount"]`), 멱등 키를 넘겨 쿠폰을 발급합니다. 예외는 잡지 않습니다. 응답이 끊기면 노드가 실패로 남아야 이 노드부터 이어 할 수 있습니다.
3. 쿠폰 ID와 처리 상태를 담은 dict를 반환합니다.

**반환값: `dict`**

| 키 | 반환할 값 |
|---|---|
| `coupon_id` | 발급된 쿠폰의 `coupon_id` |
| `status` | `"보상 완료"` |

**반환 예시**

```python
{"coupon_id": "CP-0001", "status": "보상 완료"}
```

In [ ]:
# 사람 확인 요청 노드입니다. 쿠폰은 발급하지 않고 상담원이 넘겨받도록 상태만 남깁니다.
def escalate(state: TeamState) -> dict:
    """검수가 반복해서 실패한 건을 상담원 확인 대기로 남깁니다."""
    return {"status": "사람 확인 필요"}

In [ ]:
# [작성] 같은 소포에 쿠폰이 두 번 발급되지 않게 멱등 키로 발급하세요.
def issue(state: TeamState) -> dict:
    """멱등 키로 보상 쿠폰을 발급하고 쿠폰 ID와 처리 상태를 반환합니다."""
    # 1. 소포 ID로 멱등 키를 만드세요.
    ...

    # 2. issue_coupon에 소포 ID, 보상액, 멱등 키를 넘겨 쿠폰을 발급하세요.
    ...

    # 3. coupon_id와 status를 담은 dict를 반환하세요.
    ...

In [ ]:
# [자가채점] 응답이 끊긴 뒤 다시 발급해도 쿠폰이 한 장인지 확인합니다.
reset_calls()
FAULTS["coupon_response_lost"] = True
state = {"parcel_id": "P-7001", "amount": 2000}
raised = False
try:
    issue(state)
except ConnectionError:
    raised = True
assert raised, "issue 안에서 예외를 잡지 마세요. 응답이 끊기면 노드가 실패해야 멈춘 곳부터 이어 할 수 있습니다."
assert len(COUPONS) == 1, "첫 호출에서 쿠폰 한 장이 발급되어야 합니다. issue_coupon을 호출하세요."

result = issue(state)
assert len(COUPONS) == 1, "응답이 끊긴 뒤 다시 발급하자 쿠폰이 두 장이 됐습니다. 같은 멱등 키를 issue_coupon에 넘기세요."
assert "P-7001:지연보상" in COUPON_KEYS, f"멱등 키는 '소포 ID:지연보상' 모양입니다. 지금 키: {list(COUPON_KEYS)}"
assert result == {"coupon_id": "CP-0001", "status": "보상 완료"}, f"coupon_id와 status('보상 완료')를 반환하세요. 지금: {result}"
assert COUPONS[0]["amount"] == 2000, "보상액은 state['amount']를 넘기세요."

other = issue({"parcel_id": "P-7003", "amount": 1500})
assert other["coupon_id"] == "CP-0002" and len(COUPONS) == 2, "소포가 다르면 키도 달라 새 쿠폰이 발급되어야 합니다."
reset_calls()
print("멱등 발급 확인 완료")

## 7. 실패 처리 장치를 붙여 팀 그래프를 조립하세요

1~6번에서 만든 장치를 노드에 붙입니다. `add_node`에는 재시도 정책(`retry_policy`), 시도 한 번의 시간 상한(`timeout`), 폴백 핸들러(`error_handler`)를 함께 줄 수 있습니다. `timeout`은 async 노드에만 붙일 수 있습니다(sync 노드면 compile에서 `ValueError`).

| 제공 코드 | 입력과 반환값 |
|---|---|
| `dispatch_checks(state)` | 조회 항목마다 `Send("check_worker", {"item": ..., "parcel_id": ...})`를 담은 목록을 반환합니다. |
| `make_input(parcel_id)` | 소포 ID를 받아 팀의 처음 State dict를 반환합니다. |
| `CHECK_TIMEOUT` | 조회 한 번의 시간 상한 `2`(초) |

**작성:** 팀 그래프를 조립하고 이미지로 확인하세요.

**목적:** 조회 -> 진행 판단 -> 계산 -> 작성 -> 검수 -> 발급 흐름에 재시도·타임아웃·진행 판단·폴백·핑퐁 가드를 한 그래프로 묶습니다.

**구현할 처리**
1. `TeamState`로 `team_builder`를 만들고 아래 노드 일곱 개를 등록합니다.

| 노드 이름 | 함수 | 붙일 장치 |
|---|---|---|
| `check_worker` | `check_worker` | 재시도 정책 `RETRY`, 타임아웃 `CHECK_TIMEOUT` |
| `quote` | `quote` | 폴백 핸들러 `quote_error_handler` |
| `decide_after_checks`, `writer`, `reviewer`, `escalate`, `issue` | 같은 이름의 함수 | 없음 |

2. 엣지를 등록합니다. `START`에서는 `dispatch_checks`를 조건부 엣지로 연결해 `check_worker`를 펼치고(갈 수 있는 곳 `["check_worker"]`), 고정 엣지는 `check_worker` -> `decide_after_checks`, `quote` -> `writer` -> `reviewer`, `escalate` -> `END`, `issue` -> `END`입니다. `decide_after_checks` 뒤는 3번의 `Command`가 정하므로 엣지를 등록하지 않습니다.
3. `reviewer` 뒤는 `route_after_review`를 조건부 엣지로 연결합니다. 갈 수 있는 곳은 `["writer", "escalate", "issue"]`입니다.
4. `InMemorySaver()` 체크포인터를 붙여 compile한 결과를 `team_app`에 담고, `get_graph().draw_mermaid_png()`와 `Image`, `display`로 그래프를 표시합니다.

**반환값:** 컴파일한 그래프를 `team_app` 변수에 담습니다. 이미지에 `__error_handler__quote` 노드가 함께 보입니다(핸들러를 붙이면 자동으로 생기는 노드).

**확인 기준:** 자가채점이 `writer`의 모델 체인만 고정 초안으로 바꿔 소포를 끝까지 실행합니다.
- P-7002: 배송 추적 3번 호출(2번 재시도), 날씨 2번 호출(타임아웃 뒤 재시도)이고 15초 안에 끝납니다. 규정 조회는 1번만 호출되고, `outcome`이 `degraded`이며, `failures`에 `policy`와 `quote`가 남고, 보상액이 `DEFAULT_AMOUNT`이며 쿠폰이 한 장 발급됩니다.
- P-7003: 실패 없이(`completed`) 보상액 1,500원입니다(태풍 특보로 절반).
- 배송 추적 자료가 없는 소포(채점이 `TRACKING`에서 잠시 뺍니다): `held`로 `escalate`에 가서 `"사람 확인 필요"`로 끝나고, 안내문을 쓰지 않으며 쿠폰이 없습니다.
- P-7001을 보상액이 빠진 초안으로 실행하면 `PING_PONG_LIMIT`번 반려 뒤 `"사람 확인 필요"`로 끝나고 쿠폰이 없습니다.

In [ ]:
# 조회 작업을 펼치는 함수, 처음 State, 조회 워커 타임아웃입니다.
# 조회 한 번이 이 시간(초)을 넘으면 NodeTimeoutError로 끊습니다.
CHECK_TIMEOUT = 2


def dispatch_checks(state: TeamState) -> list[Send]:
    """조회 항목마다 check_worker 하나를 Send로 띄웁니다."""
    sends = []
    for item in CHECK_ITEMS:
        sends.append(Send("check_worker", {"item": item, "parcel_id": state["parcel_id"]}))
    return sends


def make_input(parcel_id):
    """소포 ID를 받아 소포 정보 문자열과 빈 기록을 담은 처음 State를 반환합니다."""
    row = PARCELS[parcel_id]
    parcel = (f"소포 ID: {parcel_id}\n고객: {row['고객']}\n상품: {row['상품']}({row['상품 유형']})\n"
              f"구간: {row['출발']} -> {row['도착 지역']}\n약속 도착일: {row['약속 도착일']}")

    return {"parcel_id": parcel_id, "parcel": parcel, "facts": {}, "failures": {}, "outcome": "", "outcome_reason": "",
            "amount": 0, "manual_check": False, "draft": "", "rejections": [], "approved": False, "status": "", "coupon_id": ""}

In [ ]:
# [작성] 실패 처리 장치를 노드에 붙여 팀 그래프를 조립하세요.
# 1. TeamState로 team_builder를 만들고 노드 일곱 개를 등록하세요. check_worker와 quote에는 표의 장치를 붙입니다.
...

# 2. START에서 dispatch_checks로 조회 워커를 펼치고, 고정 엣지를 등록하세요. decide_after_checks 뒤는 Command가 정합니다.
...

# 3. reviewer 다음 노드는 route_after_review로 정하세요.
...

# 4. 체크포인터를 붙여 compile한 결과를 team_app에 담고 그래프를 이미지로 표시하세요.
...

In [ ]:
# [자가채점] writer의 모델 체인을 고정 초안으로 바꿔 팀을 끝까지 실행하고 실패 처리 장치를 확인합니다. 모델은 부르지 않습니다.
def good_draft(inputs):
    """채점용 고정 초안: 소포 ID, 보상액, 담당자 안내를 모두 넣습니다."""
    return AIMessage(content=f"{inputs['parcel_id']} 배송 지연 보상으로 {inputs['amount_text']} 쿠폰을 드립니다. 담당자가 다시 확인해 연락드립니다.")


def bad_draft(inputs):
    """채점용 고정 초안: 보상액이 빠져 검수에서 같은 사유로 반려됩니다."""
    return AIMessage(content=f"{inputs['parcel_id']} 배송이 늦어져 죄송합니다.")


async def run_with_draft(parcel_id, draft_fn):
    """writer_chain을 고정 초안으로 바꾸고 새 thread_id로 팀을 한 번 실행합니다."""
    fake = Mock()
    fake.invoke.side_effect = draft_fn
    config = {"configurable": {"thread_id": f"grade-{uuid4().hex[:8]}"}, "recursion_limit": 40}
    with patch.dict(writer.__globals__, {"writer_chain": fake}):
        result = await team_app.ainvoke(make_input(parcel_id), config)
    return result, fake


assert "team_app" in globals(), "compile한 결과를 team_app 변수에 담으세요."
nodes = team_app.get_graph().nodes
for name in ["check_worker", "decide_after_checks", "quote", "writer", "reviewer", "escalate", "issue"]:
    assert name in nodes, f"{name} 노드를 등록하세요."
assert "__error_handler__quote" in nodes, "quote 노드를 등록할 때 error_handler로 quote_error_handler를 붙이세요."
assert team_app.checkpointer is not None, "compile(checkpointer=InMemorySaver())로 체크포인터를 붙이세요. 멈춘 곳부터 이어 하려면 필요합니다."

# 1) 냉장 소포: 추적 재시도, 날씨 타임아웃 뒤 재시도, 규정 영구 오류 기록, 보상액 폴백
reset_calls()
started = time.time()
try:
    result, fake = await run_with_draft("P-7002", good_draft)
except (ConnectionError, NodeTimeoutError) as exc:
    raise AssertionError(f"조회 단계의 일시 오류가 그대로 올라왔습니다({type(exc).__name__}). check_worker에 재시도 정책 RETRY를 붙였는지, is_transient가 이 예외를 일시 오류로 보는지 확인하세요.")
elapsed = time.time() - started
assert CALLS["track"] == 3, f"배송 추적은 2번 끊기고 3번째에 성공합니다. check_worker의 재시도 정책을 확인하세요. 지금 {CALLS['track']}번"
assert CALLS["weather"] == 2 and elapsed < 15, f"날씨 API의 첫 호출은 응답하지 않습니다. check_worker에 CHECK_TIMEOUT 타임아웃을 붙였는지 확인하세요. 호출 {CALLS['weather']}번, {elapsed:.1f}초"
assert CALLS["policy"] == 1, "영구 오류(KeyError)는 다시 시도하지 않아야 합니다."
assert result["outcome"] != "", "outcome이 비어 있습니다. check_worker 다음에 decide_after_checks가 실행되도록 고정 엣지를 등록하세요."
assert result["outcome"] == "degraded", f"규정 조회만 실패한 소포는 degraded입니다. 3번 decide_after_checks를 확인하세요. 지금 outcome: {result['outcome']!r}"
assert result["draft"] != "", "quote가 실패한 뒤 writer가 실행되지 않았습니다. quote_error_handler가 writer로 가는 Command를 반환하는지 확인하세요."
assert "policy" in result["failures"] and "quote" in result["failures"], f"failures에 조회 실패(policy)와 보상액 계산 실패(quote)가 함께 남아야 합니다. 지금: {result['failures']}"
assert result["amount"] == DEFAULT_AMOUNT and result["manual_check"], "규정을 읽지 못하면 핸들러가 기본 보상액과 담당자 확인을 남겨야 합니다."
assert result["status"] == "보상 완료" and len(COUPONS) == 1, "통과한 안내문 뒤에는 issue가 쿠폰 한 장을 발급해야 합니다."

# 2) 제주 소포: 특보로 보상액이 절반이 됩니다.
reset_calls()
result, fake = await run_with_draft("P-7003", good_draft)
assert result["failures"] == {} and result["amount"] == 1500, f"제주 소포는 실패 없이 보상액 1,500원이어야 합니다. 지금: {result['amount']}, {result['failures']}"
assert result["outcome"] == "completed", f"실패가 없는 소포는 completed입니다. 지금 outcome: {result['outcome']!r}"

# 3) 배송 추적 자료가 없는 소포: 필수 조회 실패로 보류하고 담당자에게 넘깁니다. 빼 둔 자료는 with가 끝나면 되돌립니다.
reset_calls()
try:
    with patch.dict(TRACKING):
        del TRACKING["P-7001"]
        result, fake = await run_with_draft("P-7001", good_draft)
except KeyError as exc:
    raise AssertionError(f"배송 추적이 실패한 소포에서 quote가 실행되어 {type(exc).__name__}: {exc}가 났습니다. held이면 escalate로만 가야 합니다. decide_after_checks 뒤에 고정 엣지를 등록하지 않았는지 확인하세요.") from exc
assert result["outcome"] == "held" and result["status"] == "사람 확인 필요", f"배송 추적이 실패하면 decide_after_checks가 held로 escalate에 보내야 합니다. 지금: {result['outcome']!r}, {result['status']!r}"
assert fake.invoke.call_count == 0 and COUPONS == [], "held이면 quote·writer를 실행하지 않고 쿠폰도 발급하지 않습니다. decide_after_checks 뒤에 고정 엣지를 등록하지 않았는지 확인하세요."

# 4) 같은 반려가 반복되면 사람 확인으로 넘어가고 쿠폰은 발급하지 않습니다.
reset_calls()
result, fake = await run_with_draft("P-7001", bad_draft)
assert result["status"] == "사람 확인 필요", "같은 반려가 반복되면 reviewer 뒤에 escalate로 가야 합니다. route_after_review 연결을 확인하세요."
assert fake.invoke.call_count == PING_PONG_LIMIT and COUPONS == [], "같은 사유로 PING_PONG_LIMIT번 반려되면 더 쓰지 않고, 쿠폰도 발급하지 않습니다."
reset_calls()
print("팀 그래프 실행 확인 완료")

## 8. 멈춘 곳을 읽고 그 자리부터 이어 하세요

체크포인터가 있으면 노드가 실패해도 그때까지 끝난 단계의 결과가 thread에 남습니다. `get_state(config)`로 읽은 스냅숏의 `next`는 다음에 실행할 노드 이름 튜플, `tasks`는 그 단계의 작업 목록입니다. 작업의 `name`은 노드 이름, `error`는 실패했으면 오류 문자열이고 아니면 `None`입니다. 같은 config로 입력 없이(`None`) 다시 실행하면 처음부터가 아니라 멈춘 단계부터 실행합니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `team_app` | 7번에서 체크포인터를 붙여 컴파일한 팀 그래프 |
| `FAULTS["coupon_response_lost"]` | `True`이면 다음 쿠폰 발급 한 번의 응답이 끊깁니다. 채점 셀이 켭니다. |

**작성:** `inspect_stop(app, config)`와 async 함수 `resume_run(app, config)`를 완성하세요.

**목적:** 쿠폰 응답이 끊겨 멈춘 실행을 조회·작성·검수를 다시 하지 않고 발급 단계부터 이어 합니다.

**구현할 처리**
1. `inspect_stop`: `get_state`로 config의 thread에 남은 스냅숏을 읽습니다.
2. `inspect_stop`: `next`를 리스트로 바꾸고, `tasks` 중 `error`가 있는 작업의 `name`을 모읍니다. `error`는 문자열이므로 예외 클래스로 비교하지 말고 값이 있는지만 봅니다.
3. `inspect_stop`: `next`, `failed` 두 키의 dict를 반환합니다.
4. `resume_run`: `inspect_stop`으로 멈춘 곳을 읽어 한 줄로 출력합니다.
5. `resume_run`: 같은 config로 입력 없이 `ainvoke`를 실행(`await`)하고 그 결과(최종 State)를 반환합니다.

**반환값**

| 함수 | 반환할 값 |
|---|---|
| `inspect_stop` | `{"next": 다음 노드 이름 리스트, "failed": 실패한 작업 이름 리스트}` |
| `resume_run` | 이어 하기로 끝난 최종 State dict |

**반환 예시:** 쿠폰 응답이 끊겨 멈춘 직후의 `inspect_stop` 결과입니다.

```python
{"next": ["issue"], "failed": ["issue"]}
```

**확인 기준:** 멈춘 단계부터 다시 실행한 뒤 배송 추적·날씨·규정 호출 수가 늘지 않고, `writer`가 다시 실행되지 않으며, 쿠폰은 `CP-0001` 한 장입니다.

In [ ]:
# [작성] 멈춘 곳을 읽는 함수와 멈춘 곳부터 이어 하는 함수를 만드세요.
def inspect_stop(app, config):
    """체크포인터에 남은 마지막 상태에서 다음에 실행할 노드와 실패한 작업 이름을 반환합니다."""
    # 1. config의 thread로 마지막 상태(스냅숏)를 읽으세요.
    ...

    # 2. 다음에 실행할 노드 이름을 리스트로 만들고, error가 있는 작업의 이름을 모으세요.
    ...

    # 3. next와 failed 두 키를 가진 dict를 반환하세요.
    ...


async def resume_run(app, config):
    """멈춘 곳을 출력하고 같은 thread_id로 이어 하기를 실행해 최종 State를 반환합니다."""
    # 4. inspect_stop으로 멈춘 곳을 읽어 한 줄로 출력하세요.
    ...

    # 5. 새 입력 없이 같은 config로 이어 하기를 실행하고 결과를 반환하세요.
    ...

In [ ]:
# [자가채점] 쿠폰 응답이 끊겨 멈춘 실행을 이어 하고, 앞 단계가 다시 실행되지 않았는지 확인합니다. 모델은 부르지 않습니다.
def fixed_draft(inputs):
    """채점용 고정 초안: 검수 규칙을 모두 지킵니다."""
    return AIMessage(content=f"{inputs['parcel_id']} 배송 지연 보상으로 {inputs['amount_text']} 쿠폰을 드립니다. 담당자가 다시 확인합니다.")


fake = Mock()
fake.invoke.side_effect = fixed_draft


async def run_with_fixed_draft(inputs, config):
    """writer_chain을 고정 초안으로 바꿔 팀을 실행합니다. 쿠폰 응답이 끊겨 멈추면 True를 반환합니다."""
    with patch.dict(writer.__globals__, {"writer_chain": fake}):
        try:
            await team_app.ainvoke(inputs, config)
        except ConnectionError:
            return True
    return False


async def resume_with_fixed_draft(config):
    """writer_chain을 고정 초안으로 바꾼 채 resume_run을 실행합니다."""
    with patch.dict(writer.__globals__, {"writer_chain": fake}):
        return await resume_run(team_app, config)


reset_calls()
FAULTS["coupon_response_lost"] = True
config = {"configurable": {"thread_id": f"grade-{uuid4().hex[:8]}"}, "recursion_limit": 40}
stopped = await run_with_fixed_draft(make_input("P-7001"), config)
assert stopped, "쿠폰 응답이 끊기면 실행이 멈춰야 합니다. 6번 issue에서 예외를 잡지 않았는지 확인하세요."

# 성공한 작업과 실패한 작업이 섞인 스냅숏으로 inspect_stop만 따로 확인합니다.
from types import SimpleNamespace
mixed_snapshot = SimpleNamespace(next=("issue", "notify"), tasks=[SimpleNamespace(name="issue", error="ConnectionError('쿠폰 응답 끊김')"),
                                                                   SimpleNamespace(name="notify", error=None)])
mixed = inspect_stop(SimpleNamespace(get_state=lambda config: mixed_snapshot), {})
assert mixed == {"next": ["issue", "notify"], "failed": ["issue"]}, f"failed에는 error가 있는 작업 이름만 담으세요. 다음에 실행할 노드(next)와 다릅니다. 지금: {mixed}"

stop = inspect_stop(team_app, config)
assert stop == {"next": ["issue"], "failed": ["issue"]}, f"멈춘 곳은 issue입니다. next는 리스트로, failed에는 error가 있는 작업 이름을 담으세요. 지금: {stop}"

before = dict(CALLS)
result = await resume_with_fixed_draft(config)
for name in ["track", "weather", "policy"]:
    assert CALLS[name] == before[name], f"이어 하기에서 {name} 조회가 다시 실행됐습니다. 처음부터 다시 실행하지 말고, 새 입력 없이 같은 config로 이어 하세요."
assert fake.invoke.call_count == 1, "이어 하기에서 writer가 다시 실행됐습니다. 새 입력 없이 같은 config로 이어 하세요."
assert isinstance(result, dict) and result.get("status") == "보상 완료", "resume_run은 이어 하기로 끝난 최종 State를 반환하세요."
assert len(COUPONS) == 1 and result["coupon_id"] == "CP-0001", "이어 하기에서 쿠폰이 한 장 더 발급됐습니다. 6번 issue의 멱등 키를 확인하세요."
reset_calls()
print("이어 하기 확인 완료")

## 9. 소포 3건을 실제로 실행하고 Langfuse에 남기세요

실제 모델로 안내문을 쓰고, 실패 처리 장치가 동작한 흔적을 Langfuse에 남깁니다. 첫 쿠폰 발급의 응답이 한 번 끊기도록 준비 셀이 고장 스위치를 켜 둡니다. 이 문항에서만 모델을 호출합니다. 소포마다 날씨 타임아웃(2초)과 재시도 대기가 있어 몇 초씩 걸립니다.

소포마다 세션 id와 `thread_id`에 같은 값(`run_id`)을 넣습니다. 쿠폰 응답이 끊겨 오류로 멈춘 실행과 이어 하기 실행은 서로 다른 trace로 남지만 그 소포의 세션 하나에 모입니다. 소포 3건은 태그로 함께 걸러 봅니다.

| 제공 코드 | 입력과 반환값 |
|---|---|
| `make_config(session_id, thread_id, tags)` | 교안 02와 같은 실행 설정. 새 `CallbackHandler`가 `callbacks`에 들어 있습니다. |
| `langfuse` | `get_client()`로 만든 Langfuse 클라이언트. `get_trace_url(trace_id=...)`, `flush()`를 씁니다. |
| `run_stamp`, `TAGS` | 이번 실행 시각 문자열과 태그 `["assignment02"]` |
| `resume_run(app, config)` | 8번에서 만든 이어 하기 함수 |
| `count_calls(before)` | 실행 전 `CALLS` 복사본을 받아, 지금 `CALLS`와의 차이를 API별 dict로 반환합니다. |

**작성:** 소포 3건을 실행하고 `runs`에 결과를 담아 저장하세요.

**목적:** 실패와 이어 하기까지 포함한 실제 실행을 같은 세션의 trace로 남기고, 처리 결과를 파일로 남깁니다.

**구현할 처리**
1. 결과를 담을 `runs` dict를 만듭니다.
2. 소포마다 `run_id = f"assignment02-{run_stamp}-{parcel_id}"`를 만들고, `make_config(run_id, run_id, TAGS)`로 실행 설정을 만듭니다(세션과 `thread_id`가 같은 값).
3. 실행 전 `CALLS`를 복사해 둡니다(`dict(CALLS)`). 그다음 `team_app`을 `make_input(parcel_id)`로 실행(`await`)합니다. `ConnectionError`로 멈추면 `resume_run`으로 이어 하고 `resumed`를 `True`로 둡니다. 멈추지 않았으면 `False`입니다.
4. 실행 뒤 `count_calls`에 실행 전 복사본을 넣어 이 소포의 API별 호출 수를 구합니다.
5. `runs[parcel_id]`에 아래 아홉 키를 담고 진행 판단·상태·보상액·실패 항목·이어 하기 여부·호출 수를 한 줄로 출력합니다. trace 주소는 config의 핸들러(`config["callbacks"][0]`)의 `last_trace_id`로 `langfuse.get_trace_url`에서 얻습니다. 멈췄던 소포에는 이어 하기 실행의 trace 주소가 담깁니다.
6. `langfuse.flush()`로 전송을 마치고 `save_json("assignment02_runs.json", runs)`로 저장합니다.

**반환값: `dict`** `runs[parcel_id]`는 아래 아홉 키를 가진 dict입니다.

| 키 | 담을 값 |
|---|---|
| `outcome` | 최종 State의 `outcome`(3번 진행 판단 결과) |
| `status` | 최종 State의 `status` |
| `amount` | 최종 State의 `amount` |
| `failures` | 최종 State의 `failures` |
| `coupon_id` | 최종 State의 `coupon_id` |
| `message` | 최종 State의 `draft`(검수를 통과한 안내문) |
| `resumed` | 이어 하기를 했으면 `True` |
| `calls` | 이 소포를 처리하며 늘어난 API별 호출 수 `{"track": ..., "weather": ..., "policy": ..., "coupon": ...}` |
| `trace_url` | `langfuse.get_trace_url(trace_id=handler.last_trace_id)`로 얻은 trace 주소 |

**반환 예시:** P-7002를 실행한 경우입니다.

```python
{"P-7002": {"outcome": "degraded", "status": "보상 완료", "amount": 3000,
            "failures": {"policy": "KeyError: '1일 보상액'", "quote": "KeyError: 'policy'"},
            "coupon_id": "CP-0002", "message": "P-7002 ... 3,000원 ...",
            "resumed": False, "calls": {"track": 3, "weather": 2, "policy": 1, "coupon": 1},
            "trace_url": "https://.../project/.../traces/..."}}
```

**확인 기준:** 자가채점이 저장된 파일로 아래를 확인합니다.
- 세 소포 모두 키가 위 아홉 개이고 `calls`에 `track`, `weather`, `policy`, `coupon` 네 키가 있으며, `status`가 `"보상 완료"`이며, 안내문에 소포 ID와 보상액 표기(예: `2,000원`)가 있습니다.
- 체크포인터에 소포마다 `run_id` thread의 State가 있습니다.
- `outcome`은 P-7001 `completed`, P-7002 `degraded`, P-7003 `completed`입니다. 보상액은 P-7001 2,000원, P-7002 3,000원(기본 보상액), P-7003 1,500원입니다. P-7002의 `failures`에는 `policy`와 `quote`가 있고 안내문에 `담당자`가 있습니다.
- 이어 하기를 한 소포(`resumed`가 `True`)가 하나이고 그 쿠폰이 `CP-0001`이며, 쿠폰 ID가 `CP-0001`, `CP-0002`, `CP-0003` 한 장씩입니다.

모델 응답을 고정하거나 결과를 고쳐 쓰지 마세요. 안내문은 실행마다 다를 수 있으니 실패하면 준비 셀부터 다시 실행해 보세요.

In [ ]:
# 실행 설정 함수와 소포별 호출 수를 세는 함수입니다. make_config는 교안 02와 같고 trace 이름만 다릅니다.
def make_config(session_id, thread_id, tags, recursion_limit=30):
    """Langfuse 콜백·세션·thread_id·태그·단계 상한을 담은 그래프 실행 설정을 만듭니다."""
    return {
        # 실행마다 새 핸들러를 넣어, 실행이 끝난 뒤 그 실행의 trace id를 핸들러에서 읽습니다.
        "callbacks": [CallbackHandler()],
        "run_name": "parcel-team",
        "recursion_limit": recursion_limit,
        # thread_id는 체크포인터가 State를 저장하고, 이어 하기 때 다시 찾는 열쇠입니다.
        "configurable": {"thread_id": thread_id},
        # langfuse_로 시작하는 키는 trace의 세션·태그로 들어갑니다.
        "metadata": {"langfuse_session_id": session_id, "langfuse_tags": tags},
    }


def count_calls(before):
    """실행 전 CALLS 복사본과 지금 CALLS의 차이를 API별 dict로 반환합니다."""
    # reset_calls는 쿠폰과 고장 스위치까지 지우므로, 소포별 호출 수는 실행 전후 차이로 셉니다.
    calls = {}
    for name in CALLS:
        calls[name] = CALLS[name] - before[name]
    return calls


# Langfuse 인증을 확인하고 이번 실행의 태그를 정합니다. 이 셀부터 Langfuse 키가 필요합니다.
langfuse = get_client()
print("Langfuse 인증:", langfuse.auth_check())
# 다시 실행해도 세션·thread_id 이름이 겹치지 않게 실행 시각을 붙입니다.
run_stamp = datetime.now().strftime("%m%d-%H%M%S")
TAGS = ["assignment02"]

# 호출 횟수와 쿠폰을 비우고, 첫 쿠폰 발급의 응답이 한 번 끊기게 둡니다. 다시 실행할 때는 이 셀부터 실행하세요.
reset_calls()
FAULTS["coupon_response_lost"] = True
print("실행 시각:", run_stamp, "| 쿠폰 응답 끊김 예정:", FAULTS["coupon_response_lost"])

In [ ]:
# [작성] 소포 3건을 소포마다 자기 세션·thread로 실행하고, 멈추면 멈춘 단계부터 다시 실행한 뒤 결과를 저장하세요.
# 1. 결과를 담을 runs dict를 만드세요.
runs = {}
for parcel_id in PARCELS:
    # 2. 소포마다 run_id를 만들어 세션과 thread_id에 함께 넣고, 태그 TAGS로 실행 설정을 만드세요.
    ...

    # 3. 실행 전 CALLS를 복사해 두고 팀을 실행하세요. ConnectionError로 멈추면 resume_run으로 이어 하고 resumed를 True로 두세요.
    ...

    # 4. count_calls로 이 소포의 API별 호출 수를 구하세요.
    ...

    # 5. runs[parcel_id]에 반환값 표의 아홉 키를 담고 처리 결과와 호출 수를 한 줄로 출력하세요.
    ...

# 6. 전송을 마치고 runs를 assignment02_runs.json으로 저장하세요.
...

In [ ]:
# [자가채점] 저장된 실행 기록으로 확인합니다. 모델과 Langfuse 서버는 호출하지 않습니다.
path = output_dir / "assignment02_runs.json"
assert path.exists(), "save_json으로 runs를 assignment02_runs.json에 저장하세요."
saved = json.loads(path.read_text(encoding="utf-8"))
RETRY_HINT = "코드가 맞다면 모델이 쓴 안내문이 검수를 통과하지 못한 것일 수 있습니다. 준비 셀부터 다시 실행해 보세요."
expected_amount = {"P-7001": 2000, "P-7002": 3000, "P-7003": 1500}
expected_outcome = {"P-7001": "completed", "P-7002": "degraded", "P-7003": "completed"}
keys_needed = ["amount", "calls", "coupon_id", "failures", "message", "outcome", "resumed", "status", "trace_url"]
resumed = []
coupon_ids = []
for parcel_id in PARCELS:
    assert parcel_id in saved, f"{parcel_id} 결과가 없습니다. 소포 3건을 모두 실행하세요."
    run = saved[parcel_id]
    keys = sorted(run)
    assert keys == keys_needed, f"{parcel_id}: 키는 {keys_needed}입니다. 지금 키: {keys}"
    assert run["outcome"] == expected_outcome[parcel_id], f"{parcel_id}: outcome은 {expected_outcome[parcel_id]}이어야 합니다. 조회 실패 기록과 decide_after_checks를 확인하세요. 지금: {run['outcome']}"
    thread = {"configurable": {"thread_id": f"assignment02-{run_stamp}-{parcel_id}"}}
    assert team_app.get_state(thread).values, f"{parcel_id}: 체크포인터에 thread_id 'assignment02-{run_stamp}-{parcel_id}'의 State가 없습니다. 세션과 thread_id에 같은 run_id를 넣으세요."
    assert run["amount"] == expected_amount[parcel_id], f"{parcel_id}: 보상액은 {expected_amount[parcel_id]}원이어야 합니다. 보상액은 모델이 아니라 quote나 핸들러가 정합니다. 지금: {run['amount']}"
    assert run["status"] == "보상 완료", f"{parcel_id}: 처리 상태가 '{run['status']}'입니다. {RETRY_HINT}"
    amount_text = f"{run['amount']:,}원"
    assert parcel_id in run["message"] and amount_text in run["message"], f"{parcel_id}: message에는 검수를 통과한 안내문(draft)을 담으세요."
    langfuse_host = os.getenv("LANGFUSE_BASE_URL", "https://cloud.langfuse.com").rstrip("/")
    assert run["trace_url"].startswith(langfuse_host) and re.search(r"/traces/[0-9a-f]{32}$", run["trace_url"]), f"{parcel_id}: trace_url에는 langfuse.get_trace_url로 얻은 마지막 실행의 trace 주소를 담으세요."
    call_keys = sorted(run["calls"])
    assert call_keys == ["coupon", "policy", "track", "weather"], f"{parcel_id}: calls에는 CALLS와 같은 네 키(track, weather, policy, coupon)의 호출 수를 담으세요. 지금 키: {call_keys}"
    if run["resumed"]:
        resumed.append(parcel_id)
    coupon_ids.append(run["coupon_id"])
assert "policy" in saved["P-7002"]["failures"] and "quote" in saved["P-7002"]["failures"], "P-7002의 failures에는 policy와 quote가 있어야 합니다."
assert "담당자" in saved["P-7002"]["message"], f"P-7002 안내문에는 담당자 확인 안내가 있어야 합니다. {RETRY_HINT}"
assert len(resumed) == 1, f"쿠폰 응답은 한 번 끊기므로 이어 하기를 한 소포는 하나여야 합니다. 지금: {resumed}. 준비 셀을 다시 실행했는지, except에서 resume_run으로 이어 했는지 확인하세요."
assert saved[resumed[0]]["coupon_id"] == "CP-0001", "이어 하기를 한 소포의 쿠폰이 CP-0001이 아닙니다. 응답이 끊긴 발급과 이어 하기 실행의 발급이 같은 멱등 키를 써야 쿠폰이 한 장입니다."
coupon_ids.sort()
assert coupon_ids == ["CP-0001", "CP-0002", "CP-0003"], f"소포마다 쿠폰이 한 장씩 발급되어야 합니다. 지금: {coupon_ids}"
print("실제 실행 기록 확인 완료")

P-7002 고객 안내문입니다. 규정 조회 실패로 기본 보상액 3,000원을 먼저 드리고 담당자가 다시 확인한다는 내용이 있는지 읽어 보세요.

In [ ]:
# P-7002 고객 안내문을 읽어 봅니다.
display(Markdown(runs["P-7002"]["message"]))

## 10. Langfuse에서 실패 처리 흔적을 읽고 설명하세요

9번에서 소포마다 세션이 하나씩 생겼습니다. Langfuse의 Tracing 목록에서 태그 `assignment02`로 걸러 소포별 trace를 열고, 쿠폰 응답이 끊긴 소포는 Session ID를 눌러 세션 화면에서 답하세요.

**작성:** 서술형 답안

**목적:** 실패 처리 장치가 동작했다는 것을 관측 화면에서 무엇으로 확인하는지 설명합니다.

**설명할 내용**
1. 한 소포의 trace에서 `check_worker`의 재시도와 타임아웃은 어떻게 보였습니까? 9번에서 저장한 그 소포의 `calls`(API별 호출 수)와 trace의 `check_worker` observation 수·상태를 맞춰 설명하세요.
2. P-7002 trace에서 규정 조회 실패가 팀 전체를 멈추지 않고 안내문까지 이어진 경로를 노드 이름 순서로 쓰세요. 어느 observation이 ERROR이고 어느 것이 정상입니까? `decide_after_checks`의 Output에서 무엇을 확인했습니까?
3. 쿠폰 응답이 끊긴 소포의 세션에는 trace가 몇 개 있고, 뒤 trace에는 어떤 노드만 있습니까? 쿠폰이 한 장만 발급된 이유를 쓰세요.

정답 노트북의 모범 설명과 비교하세요.

*(여기에 확인한 화면과 판단 근거를 작성하세요.)*